In [1]:
pip install numpy tensorflow

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

# 1. Datos reales de Bankinter (Precios de cierre recientes en €)
# Ordenados de más antiguo a más reciente
precios_bkt = np.array([
    13.32, 13.38, 13.37, 13.83, 13.64, 13.67, 14.40,
    14.26, 14.49, 14.60, 14.90, 14.88
]).reshape(-1, 1)

# 2. Normalización (Crucial para que la LSTM converja)
scaler = MinMaxScaler(feature_range=(0, 1))
precios_normalizados = scaler.fit_transform(precios_bkt)

# 3. Crear ventanas de tiempo (usamos 3 días para predecir el 4º)
def crear_ventanas(datos, ventana=3):
    X, y = [], []
    for i in range(len(datos) - ventana):
        X.append(datos[i:i+ventana])
        y.append(datos[i+ventana])
    return np.array(X), np.array(y)

X, y = crear_ventanas(precios_normalizados)

# 4. Construcción del modelo
model = Sequential([
    LSTM(50, activation='relu', input_shape=(3, 1)),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')

# 5. Entrenamiento rápido
model.fit(X, y, epochs=500, verbose=0)

# 6. Predicción para el siguiente día
# Tomamos los últimos 3 precios de la lista para predecir el mañana
ultima_ventana = precios_normalizados[-3:].reshape(1, 3, 1)
prediccion_norm = model.predict(ultima_ventana, verbose=0)

# Deshacemos la normalización para ver el precio real en Euros
prediccion_final = scaler.inverse_transform(prediccion_norm)

print(f"Últimos precios: {precios_bkt[-3:].flatten()}")
print(f"Predicción Bankinter para el próximo día: {prediccion_final[0][0]:.3f}€")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Últimos precios: [14.6  14.9  14.88]
Predicción Bankinter para el próximo día: 15.049€
